In [30]:
"""
==========================================================
Passive RIS Toolkit
config.py

Global configuration for the entire toolkit.

Author : Your Name
==========================================================
"""

from dataclasses import dataclass
import numpy as np


@dataclass(frozen=True)
class RISConfig:

    # -------------------------
    # Physical constants
    # -------------------------

    c = 299792458.0

    # -------------------------
    # Operating frequency
    # -------------------------

    frequency = 5.8e9

    wavelength = c / frequency

    k = 2 * np.pi / wavelength

    # -------------------------
    # Array
    # -------------------------

    Nx = 10
    Ny = 10

    periodicity = 22.83e-3

    # -------------------------
    # Incident wave
    # -------------------------

    theta_i = 0.0
    phi_i = 0.0

    # -------------------------
    # Reflection angles
    # -------------------------

    reflection_angles = [
        0,
        30,
        45,
        60
    ]

    phi_r = 0.0

    # -------------------------
    # Unit Cells
    # -------------------------

    G1_name = "UC_G1"

    G2_name = "UC_G2"

    G1_phase = 265.0

    G2_phase = 88.0

    G1_mag = -1.95

    G2_mag = -3.81

    # -------------------------
    # Quantization
    # -------------------------

    bits = 1

    states = 2

    # -------------------------
    # File names
    # -------------------------

    element_csv = "data/ElementTable.csv"

    phase_csv = "output/csv/PhaseMap.csv"

    geometry_csv = "output/csv/GeometryMap.csv"

    vba_file = "cst/GenerateRISArray.bas"

    # -------------------------
    # Output folders
    # -------------------------

    plot_folder = "output/plots"

    report_folder = "output/reports"


CONFIG = RISConfig()

'output/plots'

In [31]:
"""
Physical constants
"""

import numpy as np

PI = np.pi

DEG2RAD = PI / 180

RAD2DEG = 180 / PI

LIGHT_SPEED = 299792458.0

EPSILON0 = 8.854187817e-12

MU0 = 4 * PI * 1e-7

ETA0 = np.sqrt(MU0 / EPSILON0)

In [32]:
import logging

logging.basicConfig(

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s"

)

logger = logging.getLogger("PassiveRIS")

In [33]:
import numpy as np

def wrap_phase(angle):

    return angle % 360


def db_to_linear(db):

    return 10 ** (db / 20)


def linear_to_db(value):

    return 20 * np.log10(value)


def deg2rad(angle):

    return np.deg2rad(angle)


def rad2deg(angle):

    return np.rad2deg(angle)

In [34]:
"""
Generate array coordinates.
"""

import numpy as np

# from config import CONFIG


class CoordinateGenerator:

    def __init__(self):

        self.Nx = CONFIG.Nx

        self.Ny = CONFIG.Ny

        self.d = CONFIG.periodicity

    def generate(self):

        x = (np.arange(self.Nx) - (self.Nx - 1) / 2) * self.d

        y = (np.arange(self.Ny) - (self.Ny - 1) / 2) * self.d

        X, Y = np.meshgrid(x, y)

        return X, Y

In [35]:
"""
==========================================================
ideal_phase.py

Computes ideal phase distribution for beam steering.

Author : Passive RIS Toolkit
==========================================================
"""

import numpy as np

# from config import CONFIG
# from coordinates import CoordinateGenerator


class IdealPhase:

    def __init__(self, theta_deg):

        self.theta_deg = theta_deg

        self.theta = np.deg2rad(theta_deg)

        self.phi = np.deg2rad(CONFIG.phi_r)

        self.k = CONFIG.k

        self.coord = CoordinateGenerator()

    def calculate(self):

        X, Y = self.coord.generate()

        phase = -self.k * (

            X * np.sin(self.theta) * np.cos(self.phi)
            +
            Y * np.sin(self.theta) * np.sin(self.phi)

        )

        phase = np.rad2deg(phase)

        phase = np.mod(phase, 360)

        return phase

In [36]:
"""
angular_distance.py
"""

import numpy as np


def angular_distance(a, b):
    """
    Circular angular distance.

    Parameters
    ----------
    a,b : degrees

    Returns
    -------
    Minimum wrapped distance
    """

    return np.abs(((a - b + 180) % 360) - 180)

In [37]:
"""
quantizer.py
"""

import numpy as np

#from config import CONFIG

# from angular_distance import angular_distance


class OneBitQuantizer:

    def __init__(self):

        self.phase1 = CONFIG.G1_phase

        self.phase2 = CONFIG.G2_phase

    def quantize(self, ideal_phase):

        d1 = angular_distance(ideal_phase, self.phase1)

        d2 = angular_distance(ideal_phase, self.phase2)

        geometry = np.where(d1 <= d2, "G1", "G2")

        quantized = np.where(d1 <= d2,
                             self.phase1,
                             self.phase2)

        return quantized, geometry

In [38]:
"""
geometry_mapper.py
"""

import pandas as pd

# from config import CONFIG


class GeometryMapper:

    def map(self,
            geometry_matrix,
            quantized_phase):

        rows = []

        Ny, Nx = geometry_matrix.shape

        for r in range(Ny):

            for c in range(Nx):

                g = geometry_matrix[r, c]

                if g == "G1":

                    rows.append({

                        "Row": r + 1,
                        "Column": c + 1,
                        "Geometry": CONFIG.G1_name,
                        "State": "G1",
                        "Phase": CONFIG.G1_phase,
                        "Magnitude": CONFIG.G1_mag

                    })

                else:

                    rows.append({

                        "Row": r + 1,
                        "Column": c + 1,
                        "Geometry": CONFIG.G2_name,
                        "State": "G2",
                        "Phase": CONFIG.G2_phase,
                        "Magnitude": CONFIG.G2_mag

                    })

        return pd.DataFrame(rows)

In [39]:
"""
lookup_table.py
"""

# from config import CONFIG


LOOKUP = {

    "G1": {

        "geometry": CONFIG.G1_name,
        "phase": CONFIG.G1_phase,
        "mag": CONFIG.G1_mag

    },

    "G2": {

        "geometry": CONFIG.G2_name,
        "phase": CONFIG.G2_phase,
        "mag": CONFIG.G2_mag

    }

}

In [40]:
"""
==============================================================
Passive RIS Toolkit
Module : element_table.py

Builds the master ElementTable for CST.

Author : Passive RIS Toolkit
==============================================================
"""

from pathlib import Path
import pandas as pd
import numpy as np

# from config import CONFIG
# from coordinates import CoordinateGenerator
# from ideal_phase import IdealPhase
# from quantizer import OneBitQuantizer


class ElementTableBuilder:
    """
    Builds the master table describing every RIS element.

    Output columns

    ID
    Row
    Column
    X(mm)
    Y(mm)
    IdealPhase(deg)
    QuantizedPhase(deg)
    State
    Geometry
    Magnitude(dB)
    """

    def __init__(self, theta_reflection):

        self.theta = theta_reflection

        self.coord = CoordinateGenerator()

        self.phase_engine = IdealPhase(theta_reflection)

        self.quantizer = OneBitQuantizer()

    # ---------------------------------------------------------

    def build(self):

        X, Y = self.coord.generate()

        ideal = self.phase_engine.calculate()

        quantized, states = self.quantizer.quantize(ideal)

        rows = []

        element_id = 1

        for r in range(CONFIG.Ny):

            for c in range(CONFIG.Nx):

                if states[r, c] == "G1":

                    geometry = CONFIG.G1_name
                    mag = CONFIG.G1_mag

                else:

                    geometry = CONFIG.G2_name
                    mag = CONFIG.G2_mag

                rows.append({

                    "ID": element_id,

                    "Row": r + 1,

                    "Column": c + 1,

                    "X_mm": X[r, c] * 1000,

                    "Y_mm": Y[r, c] * 1000,

                    "IdealPhase_deg":
                        round(float(ideal[r, c]), 3),

                    "QuantizedPhase_deg":
                        round(float(quantized[r, c]), 3),

                    "State":
                        states[r, c],

                    "Geometry":
                        geometry,

                    "Magnitude_dB":
                        mag

                })

                element_id += 1

        df = pd.DataFrame(rows)

        return df

    # ---------------------------------------------------------

    def save_csv(self,
                 filename=None):

        if filename is None:

            filename = CONFIG.element_csv

        df = self.build()

        Path(filename).parent.mkdir(
            parents=True,
            exist_ok=True
        )

        df.to_csv(
            filename,
            index=False
        )

        return df

In [41]:
"""
=============================================================
Passive RIS Toolkit

Module : export.py

Central export manager

Responsible for exporting

• ElementTable.csv
• PhaseMap.csv
• GeometryMap.csv
• Summary.txt
• Metadata.json

=============================================================
"""

from pathlib import Path
import json

import numpy as np
import pandas as pd

# from config import CONFIG


class ExportManager:

    def __init__(self):

        self.output = Path("output")
        self.csv = self.output / "csv"
        self.report = self.output / "reports"

        self.csv.mkdir(parents=True, exist_ok=True)
        self.report.mkdir(parents=True, exist_ok=True)

    ##########################################################

    def export_element_table(self,
                             dataframe):

        filename = self.csv / "ElementTable.csv"

        dataframe.to_csv(

            filename,

            index=False

        )

        print("Saved :", filename)

    ##########################################################

    def export_phase_map(self,
                         phase):

        filename = self.csv / "PhaseMap.csv"

        pd.DataFrame(phase).to_csv(

            filename,

            index=False,

            header=False

        )

        print("Saved :", filename)

    ##########################################################

    def export_geometry_map(self,
                            geometry):

        filename = self.csv / "GeometryMap.csv"

        pd.DataFrame(geometry).to_csv(

            filename,

            index=False,

            header=False

        )

        print("Saved :", filename)

    ##########################################################

    def export_numpy(self,
                     array,
                     filename):

        np.save(

            self.output / filename,

            array

        )

    ##########################################################

    def export_summary(self,
                       dataframe):

        filename = self.report / "Summary.txt"

        total = len(dataframe)

        g1 = np.sum(dataframe["State"] == "G1")

        g2 = np.sum(dataframe["State"] == "G2")

        with open(filename, "w") as f:

            f.write("PASSIVE RIS TOOLKIT\n")

            f.write("=" * 45 + "\n\n")

            f.write(f"Frequency : {CONFIG.frequency/1e9:.2f} GHz\n")

            f.write(f"Array Size : {CONFIG.Nx} x {CONFIG.Ny}\n")

            f.write(f"Periodicity : {CONFIG.periodicity*1000:.2f} mm\n")

            f.write(f"Total Elements : {total}\n\n")

            f.write(f"G1 Elements : {g1}\n")

            f.write(f"G2 Elements : {g2}\n")

            f.write(f"G1 Phase : {CONFIG.G1_phase}\n")

            f.write(f"G2 Phase : {CONFIG.G2_phase}\n")

            f.write(f"G1 Mag : {CONFIG.G1_mag} dB\n")

            f.write(f"G2 Mag : {CONFIG.G2_mag} dB\n")

        print("Saved :", filename)

    ##########################################################

    def export_metadata(self):

        filename = self.report / "Metadata.json"

        metadata = {

            "frequency_GHz":

                CONFIG.frequency / 1e9,

            "array_x":

                CONFIG.Nx,

            "array_y":

                CONFIG.Ny,

            "periodicity_mm":

                CONFIG.periodicity * 1000,

            "incident_theta":

                CONFIG.theta_i,

            "incident_phi":

                CONFIG.phi_i,

            "reflection_angles":

                CONFIG.reflection_angles,

            "G1_phase":

                CONFIG.G1_phase,

            "G2_phase":

                CONFIG.G2_phase,

            "G1_mag":

                CONFIG.G1_mag,

            "G2_mag":

                CONFIG.G2_mag

        }

        with open(filename, "w") as f:

            json.dump(

                metadata,

                f,

                indent=4

            )

        print("Saved :", filename)

    ##########################################################

    def export_all(self,
                   dataframe,
                   phase,
                   geometry):

        self.export_element_table(dataframe)

        self.export_phase_map(phase)

        self.export_geometry_map(geometry)

        self.export_summary(dataframe)

        self.export_metadata()

        print("\nExport Complete\n")

In [42]:
"""
==============================================================
Passive RIS Toolkit

verification.py

Verifies the generated ElementTable before CST automation.

Author : Passive RIS Toolkit
==============================================================
"""

from pathlib import Path
import numpy as np
import pandas as pd

# from config import CONFIG


class VerificationError(Exception):
    """Raised when verification fails."""
    pass


class Verification:

    def __init__(self):

        self.required_columns = [

            "ID",
            "Row",
            "Column",
            "X_mm",
            "Y_mm",
            "IdealPhase_deg",
            "QuantizedPhase_deg",
            "State",
            "Geometry",
            "Magnitude_dB"

        ]

    ###########################################################

    def verify_columns(self, df):

        missing = []

        for c in self.required_columns:

            if c not in df.columns:

                missing.append(c)

        if len(missing) != 0:

            raise VerificationError(

                f"Missing columns : {missing}"

            )

        print("✓ Columns verified")

    ###########################################################

    def verify_array_size(self, df):

        expected = CONFIG.Nx * CONFIG.Ny

        if len(df) != expected:

            raise VerificationError(

                f"Expected {expected} elements, found {len(df)}"

            )

        print("✓ Array size verified")

    ###########################################################

    def verify_ids(self, df):

        expected = np.arange(

            1,

            CONFIG.Nx * CONFIG.Ny + 1

        )

        if not np.array_equal(

                expected,

                df["ID"].values):

            raise VerificationError(

                "Element IDs are incorrect"

            )

        print("✓ IDs verified")

    ###########################################################

    def verify_states(self, df):

        states = set(df["State"].unique())

        allowed = {"G1", "G2"}

        if not states.issubset(allowed):

            raise VerificationError(

                "Unknown geometry state"

            )

        print("✓ States verified")

    ###########################################################

    def verify_geometry(self, df):

        valid = {

            CONFIG.G1_name,

            CONFIG.G2_name

        }

        g = set(df["Geometry"].unique())

        if not g.issubset(valid):

            raise VerificationError(

                "Invalid geometry name"

            )

        print("✓ Geometry names verified")

    ###########################################################

    def verify_phases(self, df):

        phases = set(

            np.round(

                df["QuantizedPhase_deg"],

                6

            )

        )

        valid = {

            round(CONFIG.G1_phase, 6),

            round(CONFIG.G2_phase, 6)

        }

        if not phases.issubset(valid):

            raise VerificationError(

                "Unexpected quantized phase"

            )

        print("✓ Quantized phases verified")

    ###########################################################

    def verify_duplicates(self, df):

        if df["ID"].duplicated().any():

            raise VerificationError(

                "Duplicate IDs"

            )

        if df[["Row", "Column"]].duplicated().any():

            raise VerificationError(

                "Duplicate coordinates"

            )

        print("✓ Duplicate check passed")

    ###########################################################

    def verify_coordinates(self, df):

        xs = np.sort(df["X_mm"].unique())

        ys = np.sort(df["Y_mm"].unique())

        if len(xs) != CONFIG.Nx:

            raise VerificationError(

                "Incorrect X coordinates"

            )

        if len(ys) != CONFIG.Ny:

            raise VerificationError(

                "Incorrect Y coordinates"

            )

        print("✓ Coordinates verified")

    ###########################################################

    def verify_magnitudes(self, df):

        mags = set(df["Magnitude_dB"].unique())

        valid = {

            CONFIG.G1_mag,

            CONFIG.G2_mag

        }

        if not mags.issubset(valid):

            raise VerificationError(

                "Magnitude mismatch"

            )

        print("✓ Reflection magnitudes verified")

    ###########################################################

    def verify_csv_exists(self):

        if not Path(CONFIG.element_csv).exists():

            raise VerificationError(

                "ElementTable.csv not found"

            )

        print("✓ CSV exists")

    ###########################################################

    def verify(self, df):

        print("\n")

        print("=" * 55)

        print("PASSIVE RIS TOOLKIT VERIFICATION")

        print("=" * 55)

        self.verify_columns(df)

        self.verify_array_size(df)

        self.verify_ids(df)

        self.verify_states(df)

        self.verify_geometry(df)

        self.verify_phases(df)

        self.verify_duplicates(df)

        self.verify_coordinates(df)

        self.verify_magnitudes(df)

        print("\n")

        print("✓ ALL TESTS PASSED")

        print("=" * 55)

In [43]:
"""
===========================================================
Passive RIS Toolkit

plots.py

Publication-quality plotting module.

Author : Passive RIS Toolkit
===========================================================
"""

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# from config import CONFIG


class RISPlotter:
    """
    Publication-quality plotting utilities.
    """

    def __init__(self):

        self.plot_dir = Path(CONFIG.plot_folder)

        self.plot_dir.mkdir(
            parents=True,
            exist_ok=True
        )

        plt.rcParams.update({

            "font.size": 12,

            "axes.grid": True,

            "figure.dpi": 200,

            "savefig.dpi": 600,

            "axes.labelsize": 13,

            "axes.titlesize": 14,

            "legend.fontsize": 11

        })

    #######################################################

    def _save(self,
              fig,
              filename):

        png = self.plot_dir / f"{filename}.png"

        pdf = self.plot_dir / f"{filename}.pdf"

        fig.savefig(
            png,
            bbox_inches="tight"
        )

        fig.savefig(
            pdf,
            bbox_inches="tight"
        )

        plt.close(fig)

        print(f"Saved {png}")

    #######################################################

    def phase_map(
            self,
            phase):

        """
        Phase distribution
        """

        fig, ax = plt.subplots(
            figsize=(6, 5)
        )

        im = ax.imshow(

            phase,

            origin="lower",

            cmap="twilight",

            interpolation="nearest"

        )

        ax.set_title(
            "Ideal Phase Distribution"
        )

        ax.set_xlabel("Column")

        ax.set_ylabel("Row")

        plt.colorbar(

            im,

            ax=ax,

            label="Phase (deg)"

        )

        self._save(
            fig,
            "PhaseMap"
        )

    #######################################################

    def quantized_phase_map(
            self,
            phase):

        fig, ax = plt.subplots(
            figsize=(6, 5)
        )

        im = ax.imshow(

            phase,

            origin="lower",

            cmap="viridis",

            interpolation="nearest"

        )

        ax.set_title(
            "Quantized Phase"
        )

        ax.set_xlabel(
            "Column"
        )

        ax.set_ylabel(
            "Row"
        )

        plt.colorbar(

            im,

            ax=ax,

            label="Phase (deg)"

        )

        self._save(
            fig,
            "QuantizedPhase"
        )

    #######################################################

    def geometry_map(
            self,
            geometry):

        numeric = np.where(

            geometry == "G1",

            1,

            0

        )

        fig, ax = plt.subplots(
            figsize=(6, 5)
        )

        im = ax.imshow(

            numeric,

            origin="lower",

            cmap="coolwarm",

            interpolation="nearest"

        )

        cbar = plt.colorbar(

            im,

            ax=ax,

            ticks=[0, 1]

        )

        cbar.ax.set_yticklabels(
            ["G2", "G1"]
        )

        ax.set_title(
            "Geometry Assignment"
        )

        ax.set_xlabel(
            "Column"
        )

        ax.set_ylabel(
            "Row"
        )

        self._save(
            fig,
            "GeometryMap"
        )

    #######################################################

    def coordinate_plot(
            self,
            dataframe):

        fig, ax = plt.subplots(
            figsize=(6, 6)
        )

        g1 = dataframe[
            dataframe.State == "G1"
        ]

        g2 = dataframe[
            dataframe.State == "G2"
        ]

        ax.scatter(

            g1.X_mm,

            g1.Y_mm,

            marker="s",

            s=70,

            label="G1"

        )

        ax.scatter(

            g2.X_mm,

            g2.Y_mm,

            marker="o",

            s=70,

            label="G2"

        )

        ax.set_aspect("equal")

        ax.set_xlabel(
            "X (mm)"
        )

        ax.set_ylabel(
            "Y (mm)"
        )

        ax.set_title(
            "RIS Element Coordinates"
        )

        ax.legend()

        self._save(
            fig,
            "Coordinates"
        )

In [44]:
############################################################
# Quantization Error Heatmap
############################################################

def quantization_error_map(
        self,
        error):

    fig, ax = plt.subplots(
        figsize=(6,5)
    )

    im = ax.imshow(
        error,
        origin="lower",
        cmap="inferno",
        interpolation="nearest"
    )

    ax.set_title("Quantization Error")

    ax.set_xlabel("Column")

    ax.set_ylabel("Row")

    cbar = plt.colorbar(
        im,
        ax=ax
    )

    cbar.set_label("Error (deg)")

    self._save(
        fig,
        "QuantizationError"
    )

############################################################
# Phase Histogram
############################################################

def phase_histogram(
        self,
        dataframe):

    fig, ax = plt.subplots(
        figsize=(6,4)
    )

    phase = dataframe["QuantizedPhase_deg"]

    ax.hist(

        phase,

        bins=20,

        edgecolor="black"

    )

    ax.set_xlabel("Phase (deg)")

    ax.set_ylabel("Number of Elements")

    ax.set_title("Quantized Phase Histogram")

    self._save(
        fig,
        "PhaseHistogram"
    )

############################################################
# Geometry Distribution
############################################################

def geometry_distribution(
        self,
        dataframe):

    counts = dataframe["State"].value_counts()

    fig, ax = plt.subplots(
        figsize=(5,4)
    )

    ax.bar(

        counts.index,

        counts.values

    )

    ax.set_xlabel("Geometry")

    ax.set_ylabel("Count")

    ax.set_title("Geometry Distribution")

    for x, y in zip(
            counts.index,
            counts.values):

        ax.text(

            x,

            y,

            str(y),

            ha="center",

            va="bottom"

        )

    self._save(
        fig,
        "GeometryDistribution"
    )

############################################################
# Magnitude Heatmap
############################################################

def magnitude_map(
        self,
        magnitude):

    fig, ax = plt.subplots(
        figsize=(6,5)
    )

    im = ax.imshow(

        magnitude,

        origin="lower",

        cmap="viridis",

        interpolation="nearest"

    )

    ax.set_title(
        "Reflection Magnitude"
    )

    ax.set_xlabel(
        "Column"
    )

    ax.set_ylabel(
        "Row"
    )

    cbar = plt.colorbar(

        im,

        ax=ax

    )

    cbar.set_label(
        "Magnitude (dB)"
    )

    self._save(
        fig,
        "MagnitudeMap"
    )

In [47]:
# Generate the final result plots and save them to output/plots
builder = ElementTableBuilder(theta_reflection=CONFIG.reflection_angles[0])
df = builder.build()

phase = df["IdealPhase_deg"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
quantized = df["QuantizedPhase_deg"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
geometry = df["State"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
magnitude = df["Magnitude_dB"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)

plotter = RISPlotter()
plotter.phase_map(phase)
plotter.quantized_phase_map(quantized)
plotter.geometry_map(geometry)
plotter.coordinate_plot(df)
# plotter.magnitude_map(magnitude)

Saved output\plots\PhaseMap.png
Saved output\plots\QuantizedPhase.png
Saved output\plots\GeometryMap.png
Saved output\plots\Coordinates.png


In [48]:
from pathlib import Path

for theta in CONFIG.reflection_angles:
    builder = ElementTableBuilder(theta_reflection=theta)
    df = builder.build()

    phase = df["IdealPhase_deg"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
    quantized = df["QuantizedPhase_deg"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
    geometry = df["State"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)

    angle_dir = Path(CONFIG.plot_folder) / f"theta_{theta}deg"
    angle_dir.mkdir(parents=True, exist_ok=True)

    plotter = RISPlotter()
    plotter.plot_dir = angle_dir
    plotter.phase_map(phase)
    plotter.quantized_phase_map(quantized)
    plotter.geometry_map(geometry)
    plotter.coordinate_plot(df)

Saved output\plots\theta_0deg\PhaseMap.png
Saved output\plots\theta_0deg\QuantizedPhase.png
Saved output\plots\theta_0deg\GeometryMap.png
Saved output\plots\theta_0deg\Coordinates.png
Saved output\plots\theta_30deg\PhaseMap.png
Saved output\plots\theta_30deg\QuantizedPhase.png
Saved output\plots\theta_30deg\GeometryMap.png
Saved output\plots\theta_30deg\Coordinates.png
Saved output\plots\theta_45deg\PhaseMap.png
Saved output\plots\theta_45deg\QuantizedPhase.png
Saved output\plots\theta_45deg\GeometryMap.png
Saved output\plots\theta_45deg\Coordinates.png
Saved output\plots\theta_60deg\PhaseMap.png
Saved output\plots\theta_60deg\QuantizedPhase.png
Saved output\plots\theta_60deg\GeometryMap.png
Saved output\plots\theta_60deg\Coordinates.png


In [50]:
from pathlib import Path

theta = 60
builder = ElementTableBuilder(theta_reflection=theta)
df = builder.build()

phase = df["IdealPhase_deg"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
quantized = df["QuantizedPhase_deg"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)
geometry = df["State"].to_numpy().reshape(CONFIG.Ny, CONFIG.Nx)

angle_dir = Path(CONFIG.plot_folder) / f"theta_{theta}deg"
angle_dir.mkdir(parents=True, exist_ok=True)

plotter = RISPlotter()
plotter.plot_dir = angle_dir
plotter.phase_map(phase)
plotter.quantized_phase_map(quantized)
plotter.geometry_map(geometry)
plotter.coordinate_plot(df)

Saved output\plots\theta_60deg\PhaseMap.png
Saved output\plots\theta_60deg\QuantizedPhase.png
Saved output\plots\theta_60deg\GeometryMap.png
Saved output\plots\theta_60deg\Coordinates.png


In [52]:
"""
==============================================================
Passive RIS Toolkit
reflection_model.py

Build complex reflection coefficients from measured
unit-cell responses.

Author : Passive RIS Toolkit
==============================================================
"""

from dataclasses import dataclass

import numpy as np

# from config import CONFIG


@dataclass(frozen=True)
class ReflectionState:
    name: str
    phase_deg: float
    magnitude_db: float
    amplitude: float
    gamma: complex


class ReflectionModel:

    def __init__(self):

        self.g1 = self._build_state(
            CONFIG.G1_name,
            CONFIG.G1_phase,
            CONFIG.G1_mag
        )

        self.g2 = self._build_state(
            CONFIG.G2_name,
            CONFIG.G2_phase,
            CONFIG.G2_mag
        )

    #####################################################

    @staticmethod
    def db_to_amplitude(db):

        return 10 ** (db / 20)

    #####################################################

    @staticmethod
    def build_gamma(amplitude,
                    phase_deg):

        phase = np.deg2rad(phase_deg)

        return amplitude * np.exp(
            1j * phase
        )

    #####################################################

    def _build_state(
            self,
            name,
            phase,
            mag):

        amp = self.db_to_amplitude(mag)

        gamma = self.build_gamma(
            amp,
            phase
        )

        return ReflectionState(

            name=name,

            phase_deg=phase,

            magnitude_db=mag,

            amplitude=amp,

            gamma=gamma

        )

    #####################################################

    def get_state(
            self,
            state):

        if state == "G1":

            return self.g1

        elif state == "G2":

            return self.g2

        raise ValueError(
            "Unknown state"
        )

    #####################################################

    def reflection_matrix(
            self,
            geometry):

        """
        Parameters
        ----------
        geometry

        2D matrix containing
        G1
        G2

        Returns
        -------
        complex reflection matrix
        """

        rows, cols = geometry.shape

        Gamma = np.zeros(

            (rows, cols),

            dtype=complex

        )

        for r in range(rows):

            for c in range(cols):

                if geometry[r, c] == "G1":

                    Gamma[r, c] = self.g1.gamma

                else:

                    Gamma[r, c] = self.g2.gamma

        return Gamma

    #####################################################

    def amplitude_matrix(
            self,
            geometry):

        rows, cols = geometry.shape

        amp = np.zeros(

            (rows, cols)

        )

        for r in range(rows):

            for c in range(cols):

                if geometry[r, c] == "G1":

                    amp[r, c] = self.g1.amplitude

                else:

                    amp[r, c] = self.g2.amplitude

        return amp

    #####################################################

    def phase_matrix(
            self,
            geometry):

        rows, cols = geometry.shape

        phase = np.zeros(

            (rows, cols)

        )

        for r in range(rows):

            for c in range(cols):

                if geometry[r, c] == "G1":

                    phase[r, c] = self.g1.phase_deg

                else:

                    phase[r, c] = self.g2.phase_deg

        return phase

In [55]:
"""
==============================================================
Passive RIS Toolkit
phase_error.py

Phase error analysis.

Author : Passive RIS Toolkit
==============================================================
"""

from dataclasses import dataclass
import numpy as np

# from angular_distance import angular_distance
# from reflection_model import ReflectionModel


@dataclass
class PhaseErrorResult:

    error_matrix: np.ndarray

    rms: float

    mean: float

    median: float

    std: float

    maximum: float

    minimum: float

    weighted_rms: float


class PhaseError:

    def __init__(self):

        self.model = ReflectionModel()

    #########################################################

    def compute(
            self,
            ideal_phase,
            quantized_phase):

        """
        Circular phase error
        """

        error = angular_distance(

            ideal_phase,

            quantized_phase

        )

        return error

    #########################################################

    def rms(self,
            error):

        return np.sqrt(

            np.mean(

                error ** 2

            )

        )

    #########################################################

    def weighted_rms(
            self,
            error,
            geometry):

        amp = self.model.amplitude_matrix(
            geometry
        )

        numerator = np.sum(

            amp * error ** 2

        )

        denominator = np.sum(amp)

        return np.sqrt(

            numerator / denominator

        )

    #########################################################

    def statistics(
            self,
            error):

        return {

            "mean":
                np.mean(error),

            "median":
                np.median(error),

            "std":
                np.std(error),

            "maximum":
                np.max(error),

            "minimum":
                np.min(error),

            "rms":
                self.rms(error)

        }

    #########################################################

    def analyze(
            self,
            ideal_phase,
            quantized_phase,
            geometry):

        error = self.compute(

            ideal_phase,

            quantized_phase

        )

        stats = self.statistics(
            error
        )

        wrms = self.weighted_rms(

            error,

            geometry

        )

        return PhaseErrorResult(

            error_matrix=error,

            rms=stats["rms"],

            mean=stats["mean"],

            median=stats["median"],

            std=stats["std"],

            maximum=stats["maximum"],

            minimum=stats["minimum"],

            weighted_rms=wrms

        )

    #########################################################

    def print_report(
            self,
            result):

        print()

        print("=" * 60)

        print("PHASE ERROR REPORT")

        print("=" * 60)

        print(f"Mean Error      : {result.mean:.3f} deg")

        print(f"Median Error    : {result.median:.3f} deg")

        print(f"Std Deviation   : {result.std:.3f} deg")

        print(f"Minimum Error   : {result.minimum:.3f} deg")

        print(f"Maximum Error   : {result.maximum:.3f} deg")

        print(f"RMS Error       : {result.rms:.3f} deg")

        print(f"Weighted RMS    : {result.weighted_rms:.3f} deg")

        print("=" * 60)

In [58]:
"""
==========================================================
Passive RIS Toolkit

array_factor.py

Electromagnetic Array Factor

==========================================================
"""

import numpy as np

# from config import CONFIG
# from coordinates import CoordinateGenerator
# from reflection_model import ReflectionModel


class ArrayFactor:

    def __init__(self):

        self.coord = CoordinateGenerator()

        self.model = ReflectionModel()

        self.k = CONFIG.k

    ########################################################

    def compute(
            self,
            geometry,
            theta_scan,
            phi_scan):

        """
        Parameters
        ----------

        geometry

            2D array
            containing G1/G2

        theta_scan

            degrees

        phi_scan

            degrees

        Returns

        complex array factor
        """

        X, Y = self.coord.generate()

        Gamma = self.model.reflection_matrix(
            geometry
        )

        theta = np.deg2rad(theta_scan)

        phi = np.deg2rad(phi_scan)

        u = np.sin(theta) * np.cos(phi)

        v = np.sin(theta) * np.sin(phi)

        phase = self.k * (

            X * u +

            Y * v

        )

        AF = np.sum(

            Gamma *

            np.exp(

                1j * phase

            )

        )

        return AF

    ########################################################

    def magnitude(
            self,
            geometry,
            theta,
            phi):

        AF = self.compute(

            geometry,

            theta,

            phi

        )

        return np.abs(AF)

    ########################################################

    def power(
            self,
            geometry,
            theta,
            phi):

        mag = self.magnitude(

            geometry,

            theta,

            phi

        )

        return mag ** 2

    ########################################################

    def normalized(
            self,
            geometry,
            theta_grid,
            phi_grid):

        AF = np.zeros(

            (

                len(theta_grid),

                len(phi_grid)

            )

        )

        for i, th in enumerate(theta_grid):

            for j, ph in enumerate(phi_grid):

                AF[i, j] = self.magnitude(

                    geometry,

                    th,

                    ph

                )

        AF /= AF.max()

        return AF

    ########################################################

    def db(
            self,
            geometry,
            theta_grid,
            phi_grid):

        AF = self.normalized(

            geometry,

            theta_grid,

            phi_grid

        )

        return 20 * np.log10(

            AF +

            1e-12

        )

In [62]:
"""
==============================================================
Passive RIS Toolkit

far_field.py

Far-field radiation pattern computation.

Author : Passive RIS Toolkit
==============================================================
"""

import numpy as np

# from array_factor import ArrayFactor


class FarField:

    def __init__(self):

        self.af = ArrayFactor()

    #########################################################

    @staticmethod
    def normalize(field):

        field = np.abs(field)

        maximum = np.max(field)

        if maximum == 0:

            return field

        return field / maximum

    #########################################################

    @staticmethod
    def to_db(field):

        return 20*np.log10(

            field + 1e-12

        )

    #########################################################

    def theta_cut(
            self,
            geometry,
            theta_grid,
            phi=0):

        field = []

        for theta in theta_grid:

            af = self.af.compute(

                geometry,

                theta,

                phi

            )

            field.append(

                np.abs(af)

            )

        field = np.array(field)

        field = self.normalize(field)

        return field

    #########################################################

    def theta_cut_db(
            self,
            geometry,
            theta_grid,
            phi=0):

        field = self.theta_cut(

            geometry,

            theta_grid,

            phi

        )

        return self.to_db(field)

    #########################################################

    def phi_cut(
            self,
            geometry,
            phi_grid,
            theta=0):

        field = []

        for phi in phi_grid:

            af = self.af.compute(

                geometry,

                theta,

                phi

            )

            field.append(

                np.abs(af)

            )

        field = np.array(field)

        field = self.normalize(field)

        return field

    #########################################################

    def phi_cut_db(
            self,
            geometry,
            phi_grid,
            theta=0):

        field = self.phi_cut(

            geometry,

            phi_grid,

            theta

        )

        return self.to_db(field)

    #########################################################

    def pattern_2d(
            self,
            geometry,
            theta_grid,
            phi_grid):

        pattern = np.zeros(

            (

                len(theta_grid),

                len(phi_grid)

            )

        )

        for i, theta in enumerate(theta_grid):

            for j, phi in enumerate(phi_grid):

                af = self.af.compute(

                    geometry,

                    theta,

                    phi

                )

                pattern[i, j] = np.abs(af)

        pattern = self.normalize(pattern)

        return pattern

    #########################################################

    def pattern_db(
            self,
            geometry,
            theta_grid,
            phi_grid):

        pattern = self.pattern_2d(

            geometry,

            theta_grid,

            phi_grid

        )

        return self.to_db(pattern)

In [63]:
"""
==============================================================
Passive RIS Toolkit

beam_metrics.py

Extract beam metrics from a far-field pattern.

Author : Passive RIS Toolkit
==============================================================
"""

from dataclasses import dataclass
import numpy as np


@dataclass
class BeamMetricsResult:

    peak_angle: float

    peak_level_db: float

    hpbw_deg: float

    fnbw_deg: float

    sll_db: float

    steering_error_deg: float

    directivity_est: float


class BeamMetrics:

    #########################################################

    @staticmethod
    def _peak(pattern_db):

        return int(np.argmax(pattern_db))

    #########################################################

    def peak_angle(
            self,
            theta,
            pattern_db):

        idx = self._peak(pattern_db)

        return theta[idx]

    #########################################################

    def peak_level(
            self,
            pattern_db):

        return float(np.max(pattern_db))

    #########################################################

    def hpbw(
            self,
            theta,
            pattern_db):

        peak = self._peak(pattern_db)

        threshold = pattern_db[peak] - 3

        left = peak

        while left > 0:

            if pattern_db[left] <= threshold:

                break

            left -= 1

        right = peak

        while right < len(pattern_db)-1:

            if pattern_db[right] <= threshold:

                break

            right += 1

        return theta[right] - theta[left]

    #########################################################

    def fnbw(
            self,
            theta,
            pattern_db):

        peak = self._peak(pattern_db)

        left = peak

        while left > 1:

            if pattern_db[left] < -40:

                break

            left -= 1

        right = peak

        while right < len(pattern_db)-2:

            if pattern_db[right] < -40:

                break

            right += 1

        return theta[right] - theta[left]

    #########################################################

    def sll(
            self,
            pattern_db):

        peak = self._peak(pattern_db)

        temp = pattern_db.copy()

        width = 8

        a = max(0, peak-width)

        b = min(len(temp), peak+width)

        temp[a:b] = -999

        return np.max(temp)

    #########################################################

    @staticmethod
    def steering_error(
            desired,
            measured):

        return measured-desired

    #########################################################

    @staticmethod
    def directivity(
            hpbw):

        if hpbw <= 0:

            return 0

        return 41253/(hpbw*hpbw)

    #########################################################

    def analyze(
            self,
            theta,
            pattern_db,
            desired_angle):

        peak_angle = self.peak_angle(

            theta,

            pattern_db

        )

        peak_level = self.peak_level(

            pattern_db

        )

        hpbw = self.hpbw(

            theta,

            pattern_db

        )

        fnbw = self.fnbw(

            theta,

            pattern_db

        )

        sll = self.sll(

            pattern_db

        )

        error = self.steering_error(

            desired_angle,

            peak_angle

        )

        directivity = self.directivity(

            hpbw

        )

        return BeamMetricsResult(

            peak_angle,

            peak_level,

            hpbw,

            fnbw,

            sll,

            error,

            directivity

        )

    #########################################################

    @staticmethod
    def print_report(result):

        print()

        print("="*60)

        print("BEAM METRICS")

        print("="*60)

        print(f"Peak Angle      : {result.peak_angle:.2f} deg")

        print(f"Peak Level      : {result.peak_level_db:.2f} dB")

        print(f"HPBW            : {result.hpbw_deg:.2f} deg")

        print(f"FNBW            : {result.fnbw_deg:.2f} deg")

        print(f"SLL             : {result.sll_db:.2f} dB")

        print(f"Steering Error  : {result.steering_error_deg:.2f} deg")

        print(f"Directivity Est : {result.directivity_est:.2f}")

        print("="*60)

In [66]:
"""
==============================================================
Passive RIS Toolkit
efficiency.py

Efficiency estimation for passive 1-bit RIS.

Author : Passive RIS Toolkit
==============================================================
"""

from dataclasses import dataclass
import numpy as np

# from reflection_model import ReflectionModel
# from phase_error import PhaseError


@dataclass
class EfficiencyResult:

    reflection_efficiency: float

    quantization_efficiency: float

    aperture_efficiency: float

    total_efficiency: float


class Efficiency:

    def __init__(self):

        self.model = ReflectionModel()

        self.phase = PhaseError()

    #########################################################

    def reflection_efficiency(
            self,
            geometry):

        amp = self.model.amplitude_matrix(
            geometry
        )

        power = amp ** 2

        return np.mean(power)

    #########################################################

    def quantization_efficiency(
            self,
            ideal_phase,
            quantized_phase):

        error = self.phase.compute(

            ideal_phase,

            quantized_phase

        )

        error = np.deg2rad(error)

        eta = np.mean(

            np.cos(error)

        )

        return max(0.0, eta)

    #########################################################

    @staticmethod
    def aperture_efficiency(
            geometry):

        return 1.0

    #########################################################

    def total_efficiency(
            self,
            geometry,
            ideal_phase,
            quantized_phase):

        eta_r = self.reflection_efficiency(
            geometry
        )

        eta_q = self.quantization_efficiency(
            ideal_phase,
            quantized_phase
        )

        eta_a = self.aperture_efficiency(
            geometry
        )

        return eta_r * eta_q * eta_a

    #########################################################

    def analyze(
            self,
            geometry,
            ideal_phase,
            quantized_phase):

        eta_r = self.reflection_efficiency(
            geometry
        )

        eta_q = self.quantization_efficiency(
            ideal_phase,
            quantized_phase
        )

        eta_a = self.aperture_efficiency(
            geometry
        )

        eta_total = eta_r * eta_q * eta_a

        return EfficiencyResult(

            reflection_efficiency=eta_r,

            quantization_efficiency=eta_q,

            aperture_efficiency=eta_a,

            total_efficiency=eta_total

        )

    #########################################################

    @staticmethod
    def print_report(result):

        print()

        print("=" * 60)

        print("PASSIVE RIS EFFICIENCY")

        print("=" * 60)

        print(f"Reflection Efficiency : {100*result.reflection_efficiency:.2f} %")

        print(f"Quantization Efficiency : {100*result.quantization_efficiency:.2f} %")

        print(f"Aperture Efficiency : {100*result.aperture_efficiency:.2f} %")

        print(f"Overall Efficiency : {100*result.total_efficiency:.2f} %")

        print("=" * 60)

In [67]:
"""
==============================================================
Passive RIS Toolkit

simulation_report.py

Generate a consolidated report for one RIS simulation.

Author : Passive RIS Toolkit
==============================================================
"""

from pathlib import Path
from datetime import datetime

# from config import CONFIG


class SimulationReport:

    def __init__(self):

        self.report_dir = Path(CONFIG.report_folder)

        self.report_dir.mkdir(
            parents=True,
            exist_ok=True
        )

    #########################################################

    def _header(self):

        return [
            "=" * 70,
            "PASSIVE RIS TOOLKIT REPORT",
            "=" * 70,
            f"Generated : {datetime.now()}",
            ""
        ]

    #########################################################

    def _configuration(self):

        return [

            "Configuration",

            "-------------",

            f"Frequency            : {CONFIG.frequency/1e9:.2f} GHz",

            f"Array Size           : {CONFIG.Nx} x {CONFIG.Ny}",

            f"Periodicity          : {CONFIG.periodicity*1000:.2f} mm",

            f"Incident Angle       : ({CONFIG.theta_i},{CONFIG.phi_i}) deg",

            f"Reflection Angles    : {CONFIG.reflection_angles}",

            ""
        ]

    #########################################################

    def _statistics(self, stats):

        return [

            "Geometry Distribution",

            "---------------------",

            f"G1 Elements          : {stats.g1_count}",

            f"G2 Elements          : {stats.g2_count}",

            f"G1 Percentage        : {stats.g1_percentage:.2f} %",

            f"G2 Percentage        : {stats.g2_percentage:.2f} %",

            ""
        ]

    #########################################################

    def _phase_error(self, phase):

        return [

            "Phase Error",

            "-----------",

            f"Mean                 : {phase.mean:.3f} deg",

            f"Median               : {phase.median:.3f} deg",

            f"Std                  : {phase.std:.3f} deg",

            f"RMS                  : {phase.rms:.3f} deg",

            f"Weighted RMS         : {phase.weighted_rms:.3f} deg",

            f"Maximum              : {phase.maximum:.3f} deg",

            ""
        ]

    #########################################################

    def _beam(self, beam):

        return [

            "Beam Metrics",

            "------------",

            f"Peak Angle           : {beam.peak_angle:.2f} deg",

            f"Peak Level           : {beam.peak_level_db:.2f} dB",

            f"HPBW                 : {beam.hpbw_deg:.2f} deg",

            f"FNBW                 : {beam.fnbw_deg:.2f} deg",

            f"SLL                  : {beam.sll_db:.2f} dB",

            f"Steering Error       : {beam.steering_error_deg:.2f} deg",

            f"Directivity Estimate : {beam.directivity_est:.2f}",

            ""
        ]

    #########################################################

    def _efficiency(self, eta):

        return [

            "Efficiency",

            "----------",

            f"Reflection           : {100*eta.reflection_efficiency:.2f} %",

            f"Quantization         : {100*eta.quantization_efficiency:.2f} %",

            f"Aperture             : {100*eta.aperture_efficiency:.2f} %",

            f"Overall              : {100*eta.total_efficiency:.2f} %",

            ""
        ]

    #########################################################

    def _files(self):

        return [

            "Generated Files",

            "---------------",

            "output/csv/ElementTable.csv",

            "output/csv/PhaseMap.csv",

            "output/csv/GeometryMap.csv",

            "output/plots/",

            ""

        ]

    #########################################################

    def build(

            self,

            statistics,

            phase_error,

            beam_metrics,

            efficiency):

        report = []

        report.extend(self._header())

        report.extend(self._configuration())

        report.extend(self._statistics(statistics))

        report.extend(self._phase_error(phase_error))

        report.extend(self._beam(beam_metrics))

        report.extend(self._efficiency(efficiency))

        report.extend(self._files())

        return report

    #########################################################

    def save_text(

            self,

            statistics,

            phase_error,

            beam_metrics,

            efficiency):

        report = self.build(

            statistics,

            phase_error,

            beam_metrics,

            efficiency

        )

        filename = self.report_dir / "SimulationReport.txt"

        with open(

                filename,

                "w") as f:

            for line in report:

                f.write(line + "\n")

        print("Saved :", filename)

    #########################################################

    def save_markdown(

            self,

            statistics,

            phase_error,

            beam_metrics,

            efficiency):

        report = self.build(

            statistics,

            phase_error,

            beam_metrics,

            efficiency

        )

        filename = self.report_dir / "SimulationReport.md"

        with open(

                filename,

                "w") as f:

            for line in report:

                f.write(line + "\n")

        print("Saved :", filename)

In [ ]:
Passive_RIS_Toolkit/

data/

ElementTable.csv

cst/

GenerateRISArray.bas